In [1]:
!pip install -q --upgrade kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 9.7 MB/s eta 0:00:00


In [2]:
import os
import getpass

kaggle_token = getpass.getpass("Enter your Kaggle API token: ")

os.environ["KAGGLE_API_TOKEN"] = kaggle_token

print("Kaggle API token has been set.")

Enter your Kaggle API token: ··········
Kaggle API token has been set.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

print("✓ Google Drive mounted")

Mounted at /content/drive
✓ Google Drive mounted


In [ ]:
import os

KAGGLE_DATASET = "andrewmvd/pediatric-pneumonia-chest-xray"

DOWNLOAD_DIR = "/content/kermany_external"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

print("=" * 60)
print("DOWNLOADING KERMANY DATASET")
print("=" * 60)

!kaggle datasets download \
    -d {KAGGLE_DATASET} \
    -p {DOWNLOAD_DIR}

print("\n✓ Kaggle download complete")

!ls -lh {DOWNLOAD_DIR}

DOWNLOADING KERMANY DATASET
Dataset URL: https://www.kaggle.com/datasets/andrewmvd/pediatric-pneumonia-chest-xray
License(s): Attribution 4.0 International (CC BY 4.0)
 99% 1.13G/1.14G [00:57<00:00, 21.9MB/s]

In [5]:
import zipfile
import glob
import os

zip_files = glob.glob(
    os.path.join(
        DOWNLOAD_DIR,
        "*.zip"
    )
)

print("ZIP files found:", len(zip_files))

for zip_path in zip_files:

    print("\nExtracting:", zip_path)

    with zipfile.ZipFile(
        zip_path,
        "r"
    ) as z:

        z.extractall(
            DOWNLOAD_DIR
        )

print("\n✓ Extraction complete")

for root, dirs, files_list in os.walk(DOWNLOAD_DIR):

    level = root.replace(
        DOWNLOAD_DIR,
        ""
    ).count(os.sep)

    if level <= 2:
        print(
            "  " * level +
            os.path.basename(root) +
            f" ({len(files_list)} files)"
        )

ZIP files found: 1

Extracting: /content/kermany_external/pediatric-pneumonia-chest-xray.zip

✓ Extraction complete
kermany_external (1 files)
  Pediatric Chest X-ray Pneumonia (0 files)
    test (0 files)
    train (0 files)


In [6]:
from pathlib import Path

base = Path(DOWNLOAD_DIR)

normal_files = list(
    base.rglob("test/NORMAL/*")
)

pneumonia_files = list(
    base.rglob("test/PNEUMONIA/*")
)

normal_files = [
    p for p in normal_files
    if p.suffix.lower() in [
        ".jpeg", ".jpg", ".png"
    ]
]

pneumonia_files = [
    p for p in pneumonia_files
    if p.suffix.lower() in [
        ".jpeg", ".jpg", ".png"
    ]
]

print("=" * 60)
print("KERMANY OFFICIAL TEST SPLIT")
print("=" * 60)

print("Normal images    :", len(normal_files))
print("Pneumonia images :", len(pneumonia_files))
print("Total            :",
      len(normal_files) + len(pneumonia_files))

assert len(normal_files) > 0
assert len(pneumonia_files) > 0

print("\n✓ Test images found")

KERMANY OFFICIAL TEST SPLIT
Normal images    : 234
Pneumonia images : 390
Total            : 624

✓ Test images found


In [7]:
import pandas as pd

test_rows = []

for path in normal_files:

    test_rows.append({
        "image_path": str(path),
        "true_label": 0,
        "class_name": "NORMAL"
    })

for path in pneumonia_files:

    test_rows.append({
        "image_path": str(path),
        "true_label": 1,
        "class_name": "PNEUMONIA"
    })

kermany_test_df = pd.DataFrame(
    test_rows
)

print("=" * 60)
print("KERMANY TEST DATAFRAME")
print("=" * 60)

print("Total images:", len(kermany_test_df))

print("\nClass distribution:")
print(
    kermany_test_df[
        "class_name"
    ].value_counts()
)

print("\n✓ Test dataframe ready")

KERMANY TEST DATAFRAME
Total images: 624

Class distribution:
class_name
PNEUMONIA    390
NORMAL       234
Name: count, dtype: int64

✓ Test dataframe ready


In [8]:
!pip -q install torch torchvision scikit-learn pillow

import torch
import torch.nn as nn
import torchvision.models as models

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from PIL import Image

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [9]:
kermany_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print(kermany_transform)
print("\n✓ External test transform ready")

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    Grayscale(num_output_channels=3)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

✓ External test transform ready


In [10]:
class KermanyTestDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):
        self.df = dataframe.reset_index(
            drop=True
        )
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image = Image.open(
            row["image_path"]
        ).convert("RGB")

        if self.transform:
            image = self.transform(
                image
            )

        label = torch.tensor(
            row["true_label"],
            dtype=torch.long
        )

        return (
            image,
            label
        )


kermany_test_dataset = KermanyTestDataset(
    kermany_test_df,
    transform=kermany_transform
)

kermany_test_loader = DataLoader(
    kermany_test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("=" * 60)
print("KERMANY TEST LOADER")
print("=" * 60)

images, labels = next(
    iter(kermany_test_loader)
)

print("Image shape:", images.shape)
print("Label shape:", labels.shape)

print("\n✓ DataLoader works")

KERMANY TEST LOADER
Image shape: torch.Size([32, 3, 224, 224])
Label shape: torch.Size([32])

✓ DataLoader works


In [11]:
import glob
import os

PROJECT_DIR = (
    "/content/drive/MyDrive/"
    "Pneumonia_Detection_DenseNet121"
)

model_candidates = glob.glob(
    PROJECT_DIR + "/**/*.pth",
    recursive=True
)

print("=" * 60)
print("SAVED MODEL FILES")
print("=" * 60)

for path in model_candidates:
    print(path)

SAVED MODEL FILES
/content/drive/MyDrive/Pneumonia_Detection_DenseNet121/best_densenet121_stage1.pth
/content/drive/MyDrive/Pneumonia_Detection_DenseNet121/best_densenet121_stage2.pth
/content/drive/MyDrive/Pneumonia_Detection_DenseNet121/densenet121_stage2_resume_checkpoint.pth
/content/drive/MyDrive/Pneumonia_Detection_DenseNet121/experiment_2_robust_densenet121.pth
/content/drive/MyDrive/Pneumonia_Detection_DenseNet121/experiment_2_best_resumed.pth
/content/drive/MyDrive/Pneumonia_Detection_DenseNet121/models/experiment_2_best_robust_densenet121.pth


In [13]:
import os
import torch

BASE_DIR = "/content/drive/MyDrive/Pneumonia_Detection_DenseNet121"

BASELINE_MODEL_PATH = os.path.join(
    BASE_DIR,
    "best_densenet121_stage2.pth"
)

print("=" * 60)
print("SELECTED BASELINE MODEL")
print("=" * 60)
print(BASELINE_MODEL_PATH)

assert os.path.exists(BASELINE_MODEL_PATH), "Baseline model not found!"

checkpoint = torch.load(
    BASELINE_MODEL_PATH,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCheckpoint keys:")
    print(list(checkpoint.keys()))

print("\n✓ Model file exists")
print("✓ Checkpoint loaded successfully")

SELECTED BASELINE MODEL
/content/drive/MyDrive/Pneumonia_Detection_DenseNet121/best_densenet121_stage2.pth

Checkpoint type: <class 'collections.OrderedDict'>

Checkpoint keys:
['features.conv0.weight', 'features.norm0.weight', 'features.norm0.bias', 'features.norm0.running_mean', 'features.norm0.running_var', 'features.norm0.num_batches_tracked', 'features.denseblock1.denselayer1.norm1.weight', 'features.denseblock1.denselayer1.norm1.bias', 'features.denseblock1.denselayer1.norm1.running_mean', 'features.denseblock1.denselayer1.norm1.running_var', 'features.denseblock1.denselayer1.norm1.num_batches_tracked', 'features.denseblock1.denselayer1.conv1.weight', 'features.denseblock1.denselayer1.norm2.weight', 'features.denseblock1.denselayer1.norm2.bias', 'features.denseblock1.denselayer1.norm2.running_mean', 'features.denseblock1.denselayer1.norm2.running_var', 'features.denseblock1.denselayer1.norm2.num_batches_tracked', 'features.denseblock1.denselayer1.conv2.weight', 'features.denseblo

In [14]:
# ============================================================
# LOAD BASELINE DENSENET121 FOR EXTERNAL TESTING
# ============================================================

import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_PATH = "/content/drive/MyDrive/Pneumonia_Detection_DenseNet121/best_densenet121_stage2.pth"

print("=" * 60)
print("LOADING BASELINE MODEL")
print("=" * 60)

# Recreate exact DenseNet121 architecture
model = models.densenet121(weights=None)

# Your checkpoint has 2 output classes
model.classifier = nn.Linear(
    model.classifier.in_features,
    2
)

# Load saved weights
checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
    weights_only=True
)

model.load_state_dict(checkpoint)

model = model.to(device)

# IMPORTANT: inference only
model.eval()

print("Model          : DenseNet121")
print("Checkpoint     :", MODEL_PATH)
print("Device         :", device)
print("Classifier     :", model.classifier)
print("Training mode  :", model.training)

print("\n✓ Baseline model loaded successfully")
print("✓ Weights loaded successfully")
print("✓ Model set to evaluation mode")
print("✓ NO TRAINING will be performed")

LOADING BASELINE MODEL
Model          : DenseNet121
Checkpoint     : /content/drive/MyDrive/Pneumonia_Detection_DenseNet121/best_densenet121_stage2.pth
Device         : cuda
Classifier     : Linear(in_features=1024, out_features=2, bias=True)
Training mode  : False

✓ Baseline model loaded successfully
✓ Weights loaded successfully
✓ Model set to evaluation mode
✓ NO TRAINING will be performed


In [15]:
# ============================================================
# MODEL INFERENCE SANITY CHECK
# ============================================================

dummy_input = torch.randn(2, 3, 224, 224).to(device)

with torch.no_grad():
    dummy_output = model(dummy_input)
    dummy_probability = torch.softmax(dummy_output, dim=1)

print("=" * 60)
print("MODEL SANITY CHECK")
print("=" * 60)

print("Input shape       :", dummy_input.shape)
print("Output shape      :", dummy_output.shape)
print("Probability shape :", dummy_probability.shape)
print("Probability sum   :", dummy_probability.sum(dim=1).cpu().numpy())

assert dummy_output.shape == (2, 2)
assert torch.allclose(
    dummy_probability.sum(dim=1),
    torch.ones(2).to(device),
    atol=1e-5
)

print("\n✓ Input shape correct")
print("✓ 2-class output correct")
print("✓ Softmax probabilities correct")
print("✓ Model ready for external testing")

MODEL SANITY CHECK
Input shape       : torch.Size([2, 3, 224, 224])
Output shape      : torch.Size([2, 2])
Probability shape : torch.Size([2, 2])
Probability sum   : [1. 1.]

✓ Input shape correct
✓ 2-class output correct
✓ Softmax probabilities correct
✓ Model ready for external testing


In [17]:
# ============================================================
# CELL 12 — LOAD EXPERIMENT 1 BASELINE MODEL
# ============================================================

import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# EXACT PATH FROM CELL 11
baseline_model_path = (
    "/content/drive/MyDrive/Pneumonia_Detection_DenseNet121/"
    "best_densenet121_stage2.pth"
)

print("=" * 60)
print("LOADING EXPERIMENT 1 BASELINE MODEL")
print("=" * 60)

print("Model path:")
print(baseline_model_path)

# Create DenseNet121
model = models.densenet121(weights=None)

# 2-class classifier: Normal / Pneumonia
model.classifier = nn.Linear(
    model.classifier.in_features,
    2
)

# Load trained weights
checkpoint = torch.load(
    baseline_model_path,
    map_location=device,
    weights_only=True
)

model.load_state_dict(checkpoint)

model = model.to(device)
model.eval()

print("\n✓ Experiment 1 model loaded")
print("✓ Device:", device)
print("✓ Evaluation mode:", not model.training)
print("✓ NO TRAINING")

LOADING EXPERIMENT 1 BASELINE MODEL
Model path:
/content/drive/MyDrive/Pneumonia_Detection_DenseNet121/best_densenet121_stage2.pth

✓ Experiment 1 model loaded
✓ Device: cuda
✓ Evaluation mode: True
✓ NO TRAINING


In [18]:
# ============================================================
# CELL 13 — FIND EXISTING KAGGLE TEST DATA
# ============================================================

print("=" * 60)
print("CHECKING EXISTING KAGGLE TEST DATA")
print("=" * 60)

possible_names = [
    "external_test_dataset",
    "external_test_loader",
    "test_dataset",
    "test_loader",
    "rsna_external_test_dataset",
    "rsna_external_test_loader"
]

for name in possible_names:
    if name in globals():
        obj = globals()[name]
        print(f"✓ {name} : FOUND")
        try:
            print("  Length:", len(obj))
        except:
            pass
    else:
        print(f"✗ {name} : NOT FOUND")

print("=" * 60)

CHECKING EXISTING KAGGLE TEST DATA
✗ external_test_dataset : NOT FOUND
✗ external_test_loader : NOT FOUND
✗ test_dataset : NOT FOUND
✗ test_loader : NOT FOUND
✗ rsna_external_test_dataset : NOT FOUND
✗ rsna_external_test_loader : NOT FOUND


In [20]:
# ============================================================
# CELL 12 — LOAD EXPERIMENT 1 BASELINE MODEL
# ============================================================

import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# EXACT PATH FROM CELL 11
baseline_model_path = (
    "/content/drive/MyDrive/Pneumonia_Detection_DenseNet121/"
    "best_densenet121_stage2.pth"
)

print("=" * 60)
print("LOADING EXPERIMENT 1 BASELINE MODEL")
print("=" * 60)

print("Model path:")
print(baseline_model_path)

# Create DenseNet121
model = models.densenet121(weights=None)

# 2-class classifier: Normal / Pneumonia
model.classifier = nn.Linear(
    model.classifier.in_features,
    2
)

# Load trained weights
checkpoint = torch.load(
    baseline_model_path,
    map_location=device,
    weights_only=True
)

model.load_state_dict(checkpoint)

model = model.to(device)
model.eval()

print("\n✓ Experiment 1 model loaded")
print("✓ Device:", device)
print("✓ Evaluation mode:", not model.training)
print("✓ NO TRAINING")

LOADING EXPERIMENT 1 BASELINE MODEL
Model path:
/content/drive/MyDrive/Pneumonia_Detection_DenseNet121/best_densenet121_stage2.pth

✓ Experiment 1 model loaded
✓ Device: cuda
✓ Evaluation mode: True
✓ NO TRAINING


In [21]:
# ============================================================
# CELL 12 — CHECK KERMANY DATASET STRUCTURE
# ============================================================

import os

print("=" * 60)
print("KERMANY DATASET STRUCTURE")
print("=" * 60)

for root, dirs, files in os.walk("/content"):
    if "chest" in root.lower() or "pneumonia" in root.lower() or "kermany" in root.lower():
        print(root)
        if len(files) > 0:
            print("  Files:", len(files))

KERMANY DATASET STRUCTURE
/content/drive/MyDrive/Pneumonia_Major_Project
/content/drive/MyDrive/Pneumonia_Major_Project/models
/content/drive/MyDrive/Pneumonia_Major_Project/datasets
/content/drive/MyDrive/Pneumonia_Major_Project/datasets/dataset_1_kermany
/content/drive/MyDrive/Pneumonia_Major_Project/datasets/dataset_1_kermany/val
/content/drive/MyDrive/Pneumonia_Major_Project/datasets/dataset_1_kermany/val/NORMAL
  Files: 8
/content/drive/MyDrive/Pneumonia_Major_Project/datasets/dataset_1_kermany/val/PNEUMONIA
  Files: 8
/content/drive/MyDrive/Pneumonia_Major_Project/datasets/dataset_1_kermany/test
/content/drive/MyDrive/Pneumonia_Major_Project/datasets/dataset_1_kermany/test/NORMAL
  Files: 232
/content/drive/MyDrive/Pneumonia_Major_Project/datasets/dataset_1_kermany/test/PNEUMONIA
  Files: 390
/content/drive/MyDrive/Pneumonia_Major_Project/datasets/dataset_1_kermany/train
/content/drive/MyDrive/Pneumonia_Major_Project/datasets/dataset_1_kermany/train/NORMAL
  Files: 1330
/content/

In [25]:
# ============================================================
# KERMANY EXTERNAL TEST — PATH SETUP
# ============================================================

import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

KERMANY_TEST_DIR = "/content/drive/MyDrive/Pneumonia_Major_Project/datasets/dataset_1_kermany/test"

print("=" * 60)
print("KERMANY EXTERNAL TEST SET")
print("=" * 60)

normal_dir = os.path.join(KERMANY_TEST_DIR, "NORMAL")
pneumonia_dir = os.path.join(KERMANY_TEST_DIR, "PNEUMONIA")

print("Test directory :", KERMANY_TEST_DIR)
print("NORMAL path    :", normal_dir)
print("PNEUMONIA path :", pneumonia_dir)

print("\nNORMAL images    :", len(os.listdir(normal_dir)))
print("PNEUMONIA images:", len(os.listdir(pneumonia_dir)))

assert os.path.exists(normal_dir)
assert os.path.exists(pneumonia_dir)

print("\n✓ Kermany TEST directory found")
print("✓ NORMAL folder found")
print("✓ PNEUMONIA folder found")
print("✓ TRAIN and VAL will NOT be used")

KERMANY EXTERNAL TEST SET
Test directory : /content/drive/MyDrive/Pneumonia_Major_Project/datasets/dataset_1_kermany/test
NORMAL path    : /content/drive/MyDrive/Pneumonia_Major_Project/datasets/dataset_1_kermany/test/NORMAL
PNEUMONIA path : /content/drive/MyDrive/Pneumonia_Major_Project/datasets/dataset_1_kermany/test/PNEUMONIA

NORMAL images    : 232
PNEUMONIA images: 390

✓ Kermany TEST directory found
✓ NORMAL folder found
✓ PNEUMONIA folder found
✓ TRAIN and VAL will NOT be used


In [26]:
# ============================================================
# KERMANY EXTERNAL TEST DATASET
# ============================================================

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# SAME preprocessing used by the DenseNet model
kermany_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

kermany_dataset = datasets.ImageFolder(
    root=KERMANY_TEST_DIR,
    transform=kermany_transform
)

kermany_loader = DataLoader(
    kermany_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print("=" * 60)
print("KERMANY TEST LOADER")
print("=" * 60)

print("Total images :", len(kermany_dataset))
print("Classes      :", kermany_dataset.classes)
print("Class mapping:", kermany_dataset.class_to_idx)
print("Batches      :", len(kermany_loader))

assert len(kermany_dataset) == 622 or len(kermany_dataset) == 624

print("\n✓ External test loader ready")
print("✓ No training data loaded")
print("✓ No validation data loaded")

KERMANY TEST LOADER
Total images : 622
Classes      : ['NORMAL', 'PNEUMONIA']
Class mapping: {'NORMAL': 0, 'PNEUMONIA': 1}
Batches      : 20

✓ External test loader ready
✓ No training data loaded
✓ No validation data loaded


In [27]:
# ============================================================
# KERMANY TEST BATCH CHECK
# ============================================================

images, labels = next(iter(kermany_loader))

print("=" * 60)
print("KERMANY TEST BATCH CHECK")
print("=" * 60)

print("Image shape :", images.shape)
print("Label shape :", labels.shape)
print("Image dtype :", images.dtype)
print("Image device:", images.device)

print("\nClass labels:", labels[:20].numpy())

print("\n✓ Images loaded")
print("✓ Resize = 224 × 224")
print("✓ 3-channel input")
print("✓ DataLoader working")

KERMANY TEST BATCH CHECK
Image shape : torch.Size([32, 3, 224, 224])
Label shape : torch.Size([32])
Image dtype : torch.float32
Image device: cpu

Class labels: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

✓ Images loaded
✓ Resize = 224 × 224
✓ 3-channel input
✓ DataLoader working


In [28]:
# ============================================================
# LOAD CXR-TRAINED BASELINE MODEL
# ============================================================

from torchvision import models

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

MODEL_PATH = (
    "/content/drive/MyDrive/"
    "Pneumonia_Detection_DenseNet121/"
    "best_densenet121_stage2.pth"
)

print("=" * 60)
print("LOADING CXR-TRAINED DENSENET121")
print("=" * 60)

model = models.densenet121(weights=None)

model.classifier = nn.Linear(
    model.classifier.in_features,
    2
)

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device,
    weights_only=True
)

model.load_state_dict(checkpoint)

model = model.to(device)
model.eval()

print("Model       : DenseNet121")
print("Checkpoint  :", MODEL_PATH)
print("Device      :", device)
print("Mode        :", "EVAL" if not model.training else "TRAIN")

print("\n✓ CXR-trained model loaded")
print("✓ No training")
print("✓ No fine-tuning")
print("✓ Evaluation only")

LOADING CXR-TRAINED DENSENET121
Model       : DenseNet121
Checkpoint  : /content/drive/MyDrive/Pneumonia_Detection_DenseNet121/best_densenet121_stage2.pth
Device      : cuda
Mode        : EVAL

✓ CXR-trained model loaded
✓ No training
✓ No fine-tuning
✓ Evaluation only


In [29]:
# ============================================================
# MODEL SANITY CHECK
# ============================================================

test_images, test_labels = next(iter(kermany_loader))

test_images = test_images.to(device)

with torch.no_grad():
    outputs = model(test_images)
    probabilities = torch.softmax(outputs, dim=1)

print("=" * 60)
print("MODEL SANITY CHECK")
print("=" * 60)

print("Input shape        :", test_images.shape)
print("Output shape       :", outputs.shape)
print("Probability shape  :", probabilities.shape)

print("\nFirst 5 probabilities:")
print(probabilities[:5].cpu().numpy())

assert outputs.shape[1] == 2
assert probabilities.shape[1] == 2

print("\n✓ 2-class output confirmed")
print("✓ Model accepts Kermany images")
print("✓ Ready for external inference")

MODEL SANITY CHECK
Input shape        : torch.Size([32, 3, 224, 224])
Output shape       : torch.Size([32, 2])
Probability shape  : torch.Size([32, 2])

First 5 probabilities:
[[0.9703903  0.02960963]
 [0.99750084 0.00249911]
 [0.94123065 0.05876937]
 [0.9521863  0.0478137 ]
 [0.9514215  0.04857852]]

✓ 2-class output confirmed
✓ Model accepts Kermany images
✓ Ready for external inference


In [31]:
import os
import torch

print("=" * 60)
print("CHECKING COLAB ENVIRONMENT")
print("=" * 60)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    print("WARNING: GPU is not available.")

print("\nCurrent Colab model files:")

for root, dirs, files in os.walk("/content"):
    for f in files:
        if f.endswith((".pth", ".pt", ".ckpt")):
            print(os.path.join(root, f))

print("\nKermany dataset check:")

kermany_test = "/content/kermany_external/Pediatric Chest X-ray Pneumonia/test"

print("Dataset exists:", os.path.exists(kermany_test))

if os.path.exists(kermany_test):
    print("NORMAL exists:",
          os.path.exists(os.path.join(kermany_test, "NORMAL")))
    print("PNEUMONIA exists:",
          os.path.exists(os.path.join(kermany_test, "PNEUMONIA")))

print("=" * 60)

CHECKING COLAB ENVIRONMENT
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4

Current Colab model files:
/content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/best_densenet121_stage1.pth
/content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/best_densenet121_stage2.pth
/content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/densenet121_stage2_resume_checkpoint.pth
/content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/experiment_2_robust_densenet121.pth
/content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/experiment_2_best_resumed.pth
/content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/models/experiment_2_best_robust_densenet121.pth

Kermany dataset check:
Dataset exists:

In [34]:
# ============================================================
# EXPERIMENT — KERMANY EXTERNAL VALIDATION
# INFERENCE ONLY
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("=" * 60)
print("KERMANY EXTERNAL VALIDATION — INFERENCE ONLY")
print("=" * 60)

# ------------------------------------------------------------
# 1. KERMANY TEST DATASET
# ------------------------------------------------------------

KERMANY_TEST = "/content/kermany_external/Pediatric Chest X-ray Pneumonia/test"

# IMPORTANT:
# Use the SAME normalization as your CXR model.
# DenseNet/ImageNet normalization is used here.
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

kermany_dataset = datasets.ImageFolder(
    KERMANY_TEST,
    transform=transform
)

print("Classes:", kermany_dataset.classes)
print("Class mapping:", kermany_dataset.class_to_idx)
print("Total test images:", len(kermany_dataset))

loader = DataLoader(
    kermany_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ------------------------------------------------------------
# 2. MODEL ALREADY LOADED
# ------------------------------------------------------------

model = model.to(device)
model.eval()

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 3. INFERENCE
# ------------------------------------------------------------

all_probs = []
all_preds = []
all_labels = []

print("\nRunning inference...")

with torch.no_grad():

    for images, labels in loader:

        images = images.to(device, non_blocking=True)

        outputs = model(images)

        # Handle binary-output model
        if outputs.ndim == 1:
            probs = torch.sigmoid(outputs)
        elif outputs.shape[1] == 1:
            probs = torch.sigmoid(outputs[:, 0])
        else:
            probs = torch.softmax(outputs, dim=1)[:, 1]

        preds = (probs >= 0.5).long()

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_probs = np.array(all_probs)
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

print("\nInference completed.")
print("Predictions:", len(all_preds))
print("Labels:", len(all_labels))

# ------------------------------------------------------------
# 4. METRICS
# ------------------------------------------------------------

accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, zero_division=0)
recall = recall_score(all_labels, all_preds, zero_division=0)
f1 = f1_score(all_labels, all_preds, zero_division=0)

try:
    auc = roc_auc_score(all_labels, all_probs)
except:
    auc = float("nan")

cm = confusion_matrix(all_labels, all_preds)

print("\n" + "=" * 60)
print("KERMANY EXTERNAL VALIDATION RESULTS")
print("=" * 60)

print(f"Accuracy  : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"ROC-AUC   : {auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_preds,
        target_names=kermany_dataset.classes,
        zero_division=0
    )
)

print("=" * 60)

KERMANY EXTERNAL VALIDATION — INFERENCE ONLY
Classes: ['NORMAL', 'PNEUMONIA']
Class mapping: {'NORMAL': 0, 'PNEUMONIA': 1}
Total test images: 624
Device: cuda
GPU: Tesla T4

Running inference...


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e5cc6008180>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e5cc6008180>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/


Inference completed.
Predictions: 624
Labels: 624

KERMANY EXTERNAL VALIDATION RESULTS
Accuracy  : 0.9535 (95.35%)
Precision : 0.9349
Recall    : 0.9949
F1-score  : 0.9640
ROC-AUC   : 0.9945

Confusion Matrix:
[[207  27]
 [  2 388]]

Classification Report:
              precision    recall  f1-score   support

      NORMAL       0.99      0.88      0.93       234
   PNEUMONIA       0.93      0.99      0.96       390

    accuracy                           0.95       624
   macro avg       0.96      0.94      0.95       624
weighted avg       0.96      0.95      0.95       624



In [35]:
# ============================================================
# SAVE KERMANY EXTERNAL VALIDATION RESULTS
# ============================================================

import os
import json
import pandas as pd
import numpy as np

SAVE_DIR = "/content/drive/MyDrive/Pneumonia_Major_Project/results/kermany_external_validation"
os.makedirs(SAVE_DIR, exist_ok=True)

# Metrics
metrics = {
    "dataset": "Kermany",
    "evaluation_type": "External Validation",
    "training_performed": False,
    "fine_tuning_performed": False,
    "total_samples": int(len(all_labels)),
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1_score": float(f1),
    "roc_auc": float(auc),
    "true_negative": int(cm[0, 0]),
    "false_positive": int(cm[0, 1]),
    "false_negative": int(cm[1, 0]),
    "true_positive": int(cm[1, 1])
}

with open(os.path.join(SAVE_DIR, "kermany_external_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=4)

# Classification report
report = classification_report(
    all_labels,
    all_preds,
    target_names=kermany_dataset.classes,
    zero_division=0,
    output_dict=True
)

pd.DataFrame(report).transpose().to_csv(
    os.path.join(SAVE_DIR, "classification_report.csv")
)

# Confusion matrix
pd.DataFrame(
    cm,
    index=["Actual_NORMAL", "Actual_PNEUMONIA"],
    columns=["Predicted_NORMAL", "Predicted_PNEUMONIA"]
).to_csv(
    os.path.join(SAVE_DIR, "confusion_matrix.csv")
)

# Per-image predictions
prediction_df = pd.DataFrame({
    "image_path": [kermany_dataset.samples[i][0]
                   for i in range(len(kermany_dataset))],
    "true_label": all_labels,
    "predicted_label": all_preds,
    "pneumonia_probability": all_probs
})

prediction_df.to_csv(
    os.path.join(SAVE_DIR, "per_image_predictions.csv"),
    index=False
)

print("=" * 60)
print("KERMANY RESULTS SAVED")
print("=" * 60)
print("Location:", SAVE_DIR)
print("Files:")
for f in os.listdir(SAVE_DIR):
    print("✓", f)

KERMANY RESULTS SAVED
Location: /content/drive/MyDrive/Pneumonia_Major_Project/results/kermany_external_validation
Files:
✓ kermany_external_metrics.json
✓ classification_report.csv
✓ confusion_matrix.csv
✓ per_image_predictions.csv


In [36]:
# ============================================================
# RSNA CLEANUP — DRY RUN ONLY
# NOTHING WILL BE DELETED
# ============================================================

import os

BASE = "/content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121"

DELETE_PATHS = [
    os.path.join(BASE, "GradCAM"),
    os.path.join(BASE, "external_validation_results"),
    os.path.join(BASE, "experiment_1_baseline"),
    os.path.join(BASE, "experiment_3_external_validation"),
    os.path.join(BASE, "experiment_5_uncertainty"),
    os.path.join(BASE, "experiment_6_gradcam"),
    os.path.join(BASE, "plots"),
    os.path.join(BASE, "predictions"),
    os.path.join(BASE, "logs"),
]

print("=" * 60)
print("RSNA CLEANUP — DRY RUN")
print("=" * 60)

for path in DELETE_PATHS:
    if os.path.exists(path):
        print("WOULD DELETE:", path)
    else:
        print("NOT FOUND:", path)

print("\nIMPORTANT:")
print("Model files are NOT included.")
print("Kermany dataset is NOT included.")
print("No deletion has happened.")

RSNA CLEANUP — DRY RUN
WOULD DELETE: /content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/GradCAM
WOULD DELETE: /content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/external_validation_results
WOULD DELETE: /content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/experiment_1_baseline
WOULD DELETE: /content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/experiment_3_external_validation
WOULD DELETE: /content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/experiment_5_uncertainty
WOULD DELETE: /content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/experiment_6_gradcam
WOULD DELETE: /content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/plots
W

In [37]:
import os
import shutil

BASE = "/content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121"

# These are the old RSNA-related result/output folders
DELETE_DIRS = [
    "GradCAM",
    "external_validation_results",
    "experiment_1_baseline",
    "experiment_3_external_validation",
    "experiment_5_uncertainty",
    "experiment_6_gradcam",
    "plots",
    "predictions",
    "logs",
]

print("=" * 60)
print("RSNA CLEANUP — DELETING OLD RESULTS")
print("=" * 60)

deleted = 0
missing = 0

for folder in DELETE_DIRS:
    path = os.path.join(BASE, folder)

    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"✓ DELETED: {path}")
        deleted += 1
    else:
        print(f"- NOT FOUND: {path}")
        missing += 1

print("\n" + "=" * 60)
print("CLEANUP COMPLETE")
print("=" * 60)
print(f"Folders deleted : {deleted}")
print(f"Not found       : {missing}")

print("\nIMPORTANT:")
print("✓ Model files preserved")
print("✓ Kermany dataset preserved")
print("✓ experiment_2_robust files preserved")
print("✓ No model file was deleted")

RSNA CLEANUP — DELETING OLD RESULTS
✓ DELETED: /content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/GradCAM
✓ DELETED: /content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/external_validation_results
✓ DELETED: /content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/experiment_1_baseline
✓ DELETED: /content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/experiment_3_external_validation
✓ DELETED: /content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/experiment_5_uncertainty
✓ DELETED: /content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/experiment_6_gradcam
✓ DELETED: /content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/plots
✓ DELETED

In [1]:
# ============================================================
# ROBUSTNESS EXPERIMENT — STEP 1
# Inspect existing robustness work before doing anything new
# ============================================================

import os
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB"
    "/Pneumonia_Detection_DenseNet121"
)

# Fallback to the main project directory if needed
MAIN_PROJECT = Path(
    "/content/drive/MyDrive/Pneumonia_Major_Project"
)

print("=" * 70)
print("ROBUSTNESS EXPERIMENT — EXISTING WORK INSPECTION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Check important locations
# ------------------------------------------------------------
print("\n[1] Important paths")

for p in [PROJECT_ROOT, MAIN_PROJECT]:
    print(f"{p}")
    print(f"  Exists: {p.exists()}")

# ------------------------------------------------------------
# 2. Search for robustness-related files/folders
# ------------------------------------------------------------
search_roots = [PROJECT_ROOT, MAIN_PROJECT]

keywords = [
    "robust",
    "robustness",
    "perturb",
    "brightness",
    "contrast",
    "noise",
    "blur",
    "rotation",
    "degradation",
    "augmentation"
]

matches = []

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        name_lower = path.name.lower()

        if any(k in name_lower for k in keywords):
            matches.append(path)

# Remove duplicates
matches = sorted(set(matches))

print("\n[2] Robustness-related files/folders found")

if matches:
    for p in matches:
        try:
            rel = p.relative_to(PROJECT_ROOT)
        except ValueError:
            try:
                rel = p.relative_to(MAIN_PROJECT)
            except ValueError:
                rel = p

        print("  ", rel)
else:
    print("  No robustness-related files/folders found.")

# ------------------------------------------------------------
# 3. List existing model checkpoints
# ------------------------------------------------------------
print("\n[3] Existing DenseNet checkpoints")

model_files = []

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in [".pth", ".pt"]:
            model_files.append(path)

model_files = sorted(set(model_files))

for p in model_files:
    try:
        rel = p.relative_to(PROJECT_ROOT)
    except ValueError:
        try:
            rel = p.relative_to(MAIN_PROJECT)
        except ValueError:
            rel = p

    print("  ", rel)

# ------------------------------------------------------------
# 4. Check current Colab variables
# ------------------------------------------------------------
print("\n[4] Relevant variables currently present in Colab")

candidate_variables = [
    "model",
    "baseline_model",
    "densenet121",
    "best_model",
    "device",
    "test_loader",
    "val_loader",
    "train_loader",
    "criterion",
    "optimizer"
]

for var in candidate_variables:
    if var in globals():
        obj = globals()[var]
        print(f"  {var:<20} -> {type(obj).__name__}")
    else:
        print(f"  {var:<20} -> NOT PRESENT")

print("\n" + "=" * 70)
print("INSPECTION COMPLETE")
print("=" * 70)

ROBUSTNESS EXPERIMENT — EXISTING WORK INSPECTION

[1] Important paths
/content/drive/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121
  Exists: False
/content/drive/MyDrive/Pneumonia_Major_Project
  Exists: False

[2] Robustness-related files/folders found
  No robustness-related files/folders found.

[3] Existing DenseNet checkpoints

[4] Relevant variables currently present in Colab
  model                -> NOT PRESENT
  baseline_model       -> NOT PRESENT
  densenet121          -> NOT PRESENT
  best_model           -> NOT PRESENT
  device               -> NOT PRESENT
  test_loader          -> NOT PRESENT
  val_loader           -> NOT PRESENT
  train_loader         -> NOT PRESENT
  criterion            -> NOT PRESENT
  optimizer            -> NOT PRESENT

INSPECTION COMPLETE


In [5]:
# ============================================================
# STEP 2 — INSPECT EXISTING EXPERIMENT 2 ROBUSTNESS ARTIFACTS
# ============================================================

from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path(
    "/content/drive/.shortcut-targets-by-id/"
    "1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/"
    "Pneumonia_Detection_DenseNet121"
)

print("=" * 80)
print("EXPERIMENT 2 — ROBUSTNESS ARTIFACT INSPECTION")
print("=" * 80)

print(f"\nProject root:")
print(PROJECT_ROOT)

print(f"\nExists: {PROJECT_ROOT.exists()}")

# ------------------------------------------------------------
# 1. Show top-level project structure
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("1. PROJECT STRUCTURE")
print("=" * 80)

if PROJECT_ROOT.exists():

    for item in sorted(PROJECT_ROOT.iterdir()):
        if item.is_dir():
            print(f"[DIR ]  {item.name}/")
        else:
            size_mb = item.stat().st_size / (1024 ** 2)
            print(f"[FILE]  {item.name}  ({size_mb:.2f} MB)")

# ------------------------------------------------------------
# 2. Find every Experiment 2-related artifact
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("2. ALL EXPERIMENT 2-RELATED ARTIFACTS")
print("=" * 80)

experiment2_files = []

for path in PROJECT_ROOT.rglob("*"):
    if path.is_file() and "experiment_2" in path.name.lower():
        experiment2_files.append(path)

experiment2_files = sorted(experiment2_files)

if experiment2_files:

    for path in experiment2_files:

        size_mb = path.stat().st_size / (1024 ** 2)
        modified = datetime.fromtimestamp(
            path.stat().st_mtime
        ).strftime("%Y-%m-%d %H:%M:%S")

        print("\n--------------------------------------------------")
        print(f"File:     {path.name}")
        print(f"Path:     {path}")
        print(f"Size:     {size_mb:.2f} MB")
        print(f"Modified: {modified}")

else:
    print("No Experiment 2 artifacts found.")

# ------------------------------------------------------------
# 3. Inspect likely result directories
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("3. RESULT / LOG / VISUALIZATION DIRECTORIES")
print("=" * 80)

candidate_dirs = [
    "results",
    "logs",
    "visualizations",
    "models",
    "experiment_2_robustness",
    "experiment_2",
]

for dirname in candidate_dirs:

    matches = [
        p for p in PROJECT_ROOT.rglob(dirname)
        if p.is_dir()
    ]

    if matches:
        for directory in matches:

            print(f"\n[FOUND] {directory}")

            children = list(directory.iterdir())

            if not children:
                print("   -> EMPTY")
                continue

            for child in sorted(children):
                if child.is_file():
                    size_mb = child.stat().st_size / (1024 ** 2)
                    print(
                        f"   [FILE] {child.name} "
                        f"({size_mb:.2f} MB)"
                    )
                else:
                    print(f"   [DIR ] {child.name}/")

# ------------------------------------------------------------
# 4. Search for robustness-related result files
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("4. ROBUSTNESS RESULT FILES")
print("=" * 80)

robust_keywords = [
    "robust",
    "robustness",
    "perturb",
    "brightness",
    "contrast",
    "noise",
    "blur",
    "rotation",
    "degradation",
    "corruption",
]

robust_files = []

for path in PROJECT_ROOT.rglob("*"):

    if not path.is_file():
        continue

    name = path.name.lower()

    if any(keyword in name for keyword in robust_keywords):
        robust_files.append(path)

robust_files = sorted(set(robust_files))

if robust_files:

    for path in robust_files:

        size_mb = path.stat().st_size / (1024 ** 2)

        print(
            f"{path}\n"
            f"    Size: {size_mb:.2f} MB"
        )

else:
    print("No robustness result files found.")

print("\n" + "=" * 80)
print("INSPECTION COMPLETE — NO TRAINING PERFORMED")
print("=" * 80)

EXPERIMENT 2 — ROBUSTNESS ARTIFACT INSPECTION

Project root:
/content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121

Exists: True

1. PROJECT STRUCTURE
[FILE]  FINAL_RESULTS_SUMMARY.txt  (0.00 MB)
[FILE]  best_densenet121_stage1.pth  (27.12 MB)
[FILE]  best_densenet121_stage2.pth  (27.12 MB)
[FILE]  confusion_matrix.png  (0.01 MB)
[FILE]  densenet121_stage2_resume_checkpoint.pth  (43.72 MB)
[FILE]  experiment_1_baseline_results.json  (0.00 MB)
[FILE]  final_test_results.json  (0.00 MB)
[DIR ]  models/
[FILE]  project_config.json  (0.00 MB)
[FILE]  stage2_accuracy_curve.png  (0.18 MB)
[FILE]  stage2_loss_curve.png  (0.18 MB)
[FILE]  stage2_training_history.json  (0.00 MB)
[FILE]  test_split.csv  (0.39 MB)
[FILE]  test_y_pred.npy  (0.03 MB)
[FILE]  test_y_prob.npy  (0.02 MB)
[FILE]  test_y_true.npy  (0.03 MB)
[FILE]  train_split.csv  (1.81 MB)
[FILE]  validation_split.csv  (0.39 MB)

2. ALL EXPERIMENT 2-RELATED ARTIFACTS
No Experiment 2 a

In [4]:
import os
import shutil

ROOT = "/content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121"

print("=" * 80)
print("OLD EXPERIMENT / RSNA CLEANUP")
print("=" * 80)

# ============================================================
# DELETE OLD EXPERIMENT / RSNA ARTIFACTS
# ============================================================

DELETE_ITEMS = [
    # Old Experiment 2
    "Experiment_2_Robust_Model",
    "experiment_2_robust",
    "experiment_2_robust_densenet121.pth",
    "experiment_2_robust_training_history.json",
    "experiment_2_best_resumed.pth",
    "models/experiment_2_best_robust_densenet121.pth",

    # Old RSNA / external validation
    "external_validation_results",
    "experiment_3_external_validation",
    "experiment_5_uncertainty",
    "experiment_6_gradcam",
    "GradCAM",

    # Old generated results
    "plots",
    "predictions",
    "logs",
]

deleted = []
not_found = []
failed = []

for item in DELETE_ITEMS:
    path = os.path.join(ROOT, item)

    if not os.path.exists(path):
        not_found.append(path)
        continue

    try:
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

        deleted.append(path)
        print(f"DELETED : {path}")

    except Exception as e:
        failed.append((path, str(e)))
        print(f"FAILED  : {path}")
        print(f"          {e}")

# ============================================================
# REMOVE OLD RSNA FILES THAT MAY BE INSIDE PROJECT ROOT
# ============================================================

RSNA_FILES = [
    "rsna_external_results.json",
    "rsna_y_true.npy",
    "rsna_y_pred.npy",
    "rsna_y_prob.npy",
]

for filename in RSNA_FILES:
    path = os.path.join(ROOT, filename)

    if os.path.exists(path):
        try:
            os.remove(path)
            deleted.append(path)
            print(f"DELETED : {path}")
        except Exception as e:
            failed.append((path, str(e)))
            print(f"FAILED  : {path}")
    else:
        not_found.append(path)

# ============================================================
# VERIFY IMPORTANT FILES / DATASET ARE STILL SAFE
# ============================================================

BASELINE_MODELS = [
    "best_densenet121_stage1.pth",
    "best_densenet121_stage2.pth",
    "densenet121_stage2_resume_checkpoint.pth",
]

KERMANY_ROOT = (
    "/content/drive/MyDrive/Pneumonia_Major_Project/"
    "datasets/dataset_1_kermany"
)

print("\n" + "=" * 80)
print("VERIFYING FILES THAT MUST BE KEPT")
print("=" * 80)

for filename in BASELINE_MODELS:
    path = os.path.join(ROOT, filename)
    print(
        f"{'✓ KEPT' if os.path.exists(path) else '⚠ MISSING'} : {path}"
    )

print("\nKERMANY DATASET:")
print(
    f"{'✓ KEPT' if os.path.exists(KERMANY_ROOT) else '⚠ MISSING'} : "
    f"{KERMANY_ROOT}"
)

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("CLEANUP COMPLETE")
print("=" * 80)

print(f"Deleted    : {len(deleted)}")
print(f"Not found : {len(not_found)}")
print(f"Failed     : {len(failed)}")

if failed:
    print("\nFAILED ITEMS:")
    for path, error in failed:
        print(path)
        print(error)

print("\n✓ Old experiment artifacts removed")
print("✓ Old RSNA artifacts removed")
print("✓ Baseline model preserved")
print("✓ Kermany dataset preserved")
print("\nREADY FOR FRESH EXPERIMENT 2 — KERMANY DATASET")
print("=" * 80)

OLD EXPERIMENT / RSNA CLEANUP
DELETED : /content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/Experiment_2_Robust_Model
DELETED : /content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/experiment_2_robust
DELETED : /content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/experiment_2_robust_densenet121.pth
DELETED : /content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/experiment_2_robust_training_history.json
DELETED : /content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/experiment_2_best_resumed.pth
DELETED : /content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBafr7CB/Pneumonia_Detection_DenseNet121/models/experiment_2_best_robust_densenet121.pth
DELETED : /content/drive/.shortcut-targets-by-id/1q-s9UO9qA7XMIa6RJEPo1uh8eBaf